# Quickstart: learn a landscape relationship

How can two landscape covariates help explain genetic dissimilarity among sampled populations?
This first analysis prepares a small synthetic landscape, trains a conductance model, inspects
pair-aligned predictions, and saves both a fitted model and a resumable checkpoint.

> **Note:** We use one region and one species. No species labels are needed. The numerical examples are
> small enough for a CPU; their training budget illustrates the workflow rather than establishes
> convergence or biological evidence. See the [maximum-likelihood population effects (MLPE) workflow](mlpe_workflow.md) for population
> dependence, fresh calibration, and assessment on separate populations.

In [ ]:
from dataclasses import replace
from itertools import combinations
from pathlib import Path
from tempfile import TemporaryDirectory

import coordax as cx
import jax
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from ilg_toolkit import (
    Log1pLoss,
    PairwiseObservations,
    RegionBatch,
    TargetSpec,
    TrainingConfig,
    calibrate,
    load_checkpoint,
    load_model,
    save_checkpoint,
    save_model,
    train,
)
from ilg_toolkit.models import LogLinearConductance

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(
    {"figure.figsize": (7, 3.5), "axes.spines.top": False, "axes.spines.right": False}
)

## Prepare landscape features and observations

A `RegionBatch` describes located sampling units and landscape features. Row and column
positions refer to raster cells, while feature names establish their meanings and order.
The two covariates below are standardized synthetic canopy and moisture indices.

> **Note:** `PairwiseObservations` holds measured relationships and a `TargetSpec`. We declare an
> identity-transformed dissimilarity in index units. An absent measurement would be omitted
> from `from_pairs`; it should never be replaced with a zero.

In [ ]:
row, column = np.mgrid[-1:1:4j, -1:1:4j]
features = np.stack((column, row + 0.3 * column**2), axis=-1)
labels = ("west", "north", "east", "south")
positions = np.array([[0, 0], [0, 3], [3, 3], [3, 0]])
region = RegionBatch(
    "valley",
    cx.field(features, "row", "column", "feature"),
    labels,
    positions,
    feature_names=("canopy", "moisture"),
)
observations = PairwiseObservations.from_pairs(
    tuple(combinations(labels, 2)),
    [0.8, 1.3, 0.7, 0.9, 1.2, 0.6],
    target=TargetSpec("synthetic genetic dissimilarity", units="index"),
    sampling_unit_ids=labels,
)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for index, ax in enumerate(axes):
    image = ax.imshow(features[..., index], cmap="viridis")
    ax.scatter(positions[:, 1], positions[:, 0], color="white", edgecolor="black")
    for label, (r, c) in zip(labels, positions, strict=True):
        ax.annotate(label, (c, r), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set(title=region.feature_names[index], xlabel="column", ylabel="row")
    fig.colorbar(image, ax=ax, label="standardized feature")
fig.tight_layout()
plt.show()

## Train and continue a single run

> **Note:** `LogLinearConductance` gives each covariate an interpretable coefficient. With `patch_size=1`,
> each raster cell becomes one conductance cell. Larger patches change the represented
> landscape; they are a modeling choice, not just a memory setting.

`Log1pLoss` compares log1p ecological distances with log1p model-scale targets. This use of
log1p belongs to the loss; the observation target transform is a separate choice. Here the
target transform is the identity. Direct calibration therefore returns ecological distances
in the declared target units without estimating additional statistical coefficients.

First run four Adam updates, then save and continue to eight **total** updates. All other
configuration and inputs stay the same. A zero-step run would initialize and evaluate the
model without any updates.

In [ ]:
distance_model = LogLinearConductance(2, patch_size=1)
config = TrainingConfig(loss=Log1pLoss(), steps=4, eval_every=2, learning_rate=0.03, seed=4)
state = train(region, observations, distance_model=distance_model, config=config)

workspace = TemporaryDirectory()
artifact_dir = Path(workspace.name)
checkpoint_path = artifact_dir / "training.eqx"
save_checkpoint(checkpoint_path, state)
template = LogLinearConductance(2, patch_size=1)
restored_state = load_checkpoint(checkpoint_path, distance_model=template)
continued = train(
    region,
    observations,
    state=restored_state,
    config=replace(restored_state.config, steps=8),
)
print(f"Checkpoint step: {restored_state.step}; continued step: {continued.step}")
print(f"Selected evaluated step: {continued.selected_step}")
plt.plot(
    [record.step for record in continued.history],
    [record.training_loss for record in continued.history],
    marker="o",
)
plt.xlabel("Adam updates")
plt.ylabel("Training Log1pLoss")
plt.title("Optimization on these six measured pairs")
plt.tight_layout()
plt.show()

> **Note:** The loss curve describes fit to the supplied observations. A decrease is not evidence of
> generalization. With no built-in validation, `selected_model` is the final evaluated model.
> `latest_model`, optimizer state, loss state, and random state are the pieces used to continue
> training. A validation-selected model may belong to an earlier step.

Exact continuation assumes unchanged data, library versions, device, and execution settings.
The total step budget can increase. Checkpoints taken from `on_step` retain the evaluation
history accumulated at that point; extending a completed run can add a terminal evaluation
that was absent from a differently scheduled run. See the [persistence API](../api/persistence.md).

## Inspect the learned relationship

Conductance describes how readily the modeled landscape permits movement. Genetic predictions
describe pairwise relationships on the declared target scale. A bare distance model returns
ecological distances; a landscape genetics model includes the genetic calibration.

> **Note:** `calibrate` below shows the frozen-model operation for the parameter-free direct loss.
> It needs no iteration budget. For MLPE, the same operation estimates a fresh relationship,
> including both variance components; the [MLPE workflow](mlpe_workflow.md) develops that case.

In [ ]:
model = calibrate(region, observations, continued.selected_model.distance_model, loss=Log1pLoss())
surface = model.distance_model.conductance_surface(region)
prediction = model.predict(region, pairs=observations.observed_pairs)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
image = axes[0].imshow(surface, cmap="viridis")
axes[0].set(title="Learned conductance", xlabel="column", ylabel="row")
fig.colorbar(image, ax=axes[0])
axes[1].scatter(observations.observed_values, prediction.values)
axes[1].plot([0, 1.5], [0, 1.5], color="gray", linestyle="--")
axes[1].set(
    xlabel="Observed dissimilarity (index)",
    ylabel="Predicted relationship (index)",
    title="Fit to the supplied observations",
)
fig.tight_layout()
plt.show()

rows = ["| Pair | Observed | Predicted |", "| --- | ---: | ---: |"]
for pair, observed, predicted in zip(
    prediction.pairs, observations.observed_values, prediction.values, strict=True
):
    rows.append(f"| {' – '.join(pair)} | {float(observed):.3f} | {float(predicted):.3f} |")
display(Markdown("\n".join(rows)))

## Request only the pairs needed

> **Note:** Predictions retain the requested pair order and orientation. `as_matrix()` is a convenience
> view: the diagonal and every unrequested pair are `NaN`. Requesting a sparse vector avoids
> allocating the output matrix; calling `as_matrix()` allocates it. Resistance calculation may
> still use its sampling-unit solver matrices, so output sparsity is not a promise that every
> internal calculation is sparse.

In [ ]:
query_pairs = [("east", "west"), ("north", "south")]
query = model.predict(region, pairs=query_pairs)
distances = model.distance_model.predict(region, pairs=query_pairs)
print("Requested pair order:", query.pairs)
print("Ecological distances:", np.asarray(distances.values))
print("Genetic predictions (index):", np.asarray(query.values))
print("Sparse request arranged as a matrix:\n", np.asarray(query.as_matrix()))

## Save a fitted model for later predictions

A model file retains target semantics, feature meanings, species order, fitted calibration,
and numerical parameters. Loading uses a caller-constructed distance-model template with
matching class, static settings, shapes, and dtypes. Enable float64 before constructing and
loading models that require it. Model files support prediction; a checkpoint also contains
the optimizer and random state needed for continuation.

The temporary directory keeps this tutorial self-contained. In an analysis, choose a persistent
directory whose parent already exists.

In [ ]:
model_path = artifact_dir / "valley-model.eqx"
save_model(model_path, model)
restored_model = load_model(model_path, distance_model=LogLinearConductance(2, patch_size=1))
print("Saved files:", sorted(path.name for path in artifact_dir.iterdir()))
print("Reloaded target:", restored_model.target)
print("Reloaded predictions:", np.asarray(restored_model.predict(region, pairs=query_pairs).values))
workspace.cleanup()

## What this analysis establishes

> **Note:** We obtained a reproducible fitted relationship and usable prediction and continuation files.
> The six observations, tiny landscape, and short optimization budget do not establish a
> biological effect, model identifiability, or predictive accuracy. Feature scaling, landscape
> extent, and sampling design all deserve study-specific decisions.

Next, inspect the [isolation-by-distance reference](ibd.md), then use the
[MLPE workflow](mlpe_workflow.md) to account for population dependence and compare frozen
landscape distances with a geographic baseline on separate populations.